In [3]:
import json, math, pathlib, textwrap, zipfile, shutil
import numpy as np, pandas as pd
from scipy.stats import binomtest

FOLDER_NAME = "cot_sweep"                  # "cot_sweep" for the full sweep, "cot_sweep_mini" for the mini run
GOLD_PATH = "/content/GOLD54_escalation_failure_df.jsonl"
MAX_TOKENS = 1500                          # budget used in the sweep
UNZIP_TO = pathlib.Path("/content/cot_unzipped")

# 1. upload the zip (skipped if a .zip is already sitting in /content)
zips = sorted(pathlib.Path("/content").glob("*.zip"))
if not zips:
    from google.colab import files
    up = files.upload()                    # pick e.g. cot_sweep_both.zip
    zips = [pathlib.Path("/content") / n for n in up if n.endswith(".zip")]
assert zips, "no .zip found or uploaded"
zip_path = zips[-1]
print("using", zip_path)

# 2. unzip (fresh folder each run, so a re-upload never mixes old and new files)
shutil.rmtree(UNZIP_TO, ignore_errors=True); UNZIP_TO.mkdir()
with zipfile.ZipFile(zip_path) as z:
    z.extractall(UNZIP_TO)

# 3. find the folder that holds the sweep files, wherever it sits inside the zip
cands = [p for p in UNZIP_TO.rglob(FOLDER_NAME) if p.is_dir() and any(p.glob("*__*.jsonl"))]
if not cands:                              # zip of the bare files, or a differently named folder
    cands = [p for p in [UNZIP_TO, *UNZIP_TO.rglob("*")] if p.is_dir() and any(p.glob("*__*.jsonl"))]
assert cands, f"no *__*.jsonl files found inside {zip_path.name}"
FOLDER = cands[0]
print("sweep folder:", FOLDER, "| other folders with jsonl:", [str(p) for p in cands[1:]] or "none")

# 4. gold file and parser
gold = [json.loads(l) for l in open(GOLD_PATH, encoding="utf-8")]
for i, g in enumerate(gold):               # this file has no item_id and calls the escalation field esc_answer_idx
    g.setdefault("item_id", i)
    g.setdefault("esc", g.get("esc_answer_idx"))
by_item = {g["item_id"]: g for g in gold}

# 5. read and check completeness
raw = []
for f in sorted(FOLDER.glob("*__*.jsonl")):
    raw += [json.loads(l) for l in open(f, encoding="utf-8")]
cnt = pd.DataFrame(raw).groupby(["model", "cond"]).item.agg(n="count", unique="nunique").unstack()
print(len(raw), "raw responses from", FOLDER)
print(cnt.to_string())
short = cnt[(cnt[("n", "letter")] < len(gold)) | (cnt[("n", "cot")] < len(gold))]
print("\nincomplete models (re-run run_sweep in the sweep notebook):", list(short.index) if len(short) else "none, every file has all", len(gold), "items")

using /content/cot_sweep_before_rerun_1023_0832.zip
sweep folder: /content/cot_unzipped/cot_sweep | other folders with jsonl: none
1404 raw responses from /content/cot_unzipped/cot_sweep
                        n        unique       
cond                  cot letter    cot letter
model                                         
claude_haiku_5_5       54     54     54     54
deepseek_v4_1_flash    54     54     54     54
gpt_6_luna             54     54     54     54
grok_4_7               54     54     54     54
legacy_claude_3_haiku  54     54     54     54
legacy_deepseek        54     54     54     54
legacy_gemini_flash    54     54     54     54
legacy_gpt_4o_mini     54     54     54     54
legacy_llama3_70b      54     54     54     54
legacy_mistral_small   54     54     54     54
legacy_qwen_7b         54     54     54     54
mistral_large_4        54     54     54     54
qwen3_8_omni_flash     54     54     54     54

incomplete models (re-run run_sweep in the sweep notebook): 

In [5]:
"""Unified answer parser for the escalation-failure evaluation (v2).

One deterministic, model-agnostic cascade replacing the per-model parsers of the original notebooks.
Every response gets exactly one status: explicit | final_line | option_text | truncated | unparseable.
Only the first three yield a letter; truncated/unparseable are reported separately from clinical failure.
"""
import json
import re

LETTERS = "ABCDE"
L = r"([A-E])"

# Ordered, most specific first.
EXPLICIT_PATTERNS = [
    re.compile(r"\\boxed\{\s*\(?" + L + r"\)?\s*\}", re.I),
    re.compile(r"answer\s+is\s*:?\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:final|correct|best|chosen)\s+(?:answer|choice|option)\s*(?:is)?\s*:?\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:^|\n)\s*\**\s*answer\s*:\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:is|be|choose|select|pick)\s*:?\s*\n+\s*\**\(?" + L + r"\)?[\)\.\*:]", re.I),
    re.compile(r"\*\*\s*\(?" + L + r"\)?\s*[\)\.:]?\s*\*\*"),
    # conclusion sentence naming a lettered option: "...management is D) Thiamine", "best choice is (B)"
    re.compile(r"(?:management|step|choice|option|treatment|answer|intervention)\s+(?:is|would be|should be)\s*:?\s*\**\(?" + L + r"\)\s", re.I),
]
FINAL_LINE = re.compile(r"^\W*\(?" + L + r"\)?\s*(?:[\.\):\-]\s*.*)?\W*$", re.S)


def _last_line(text):
    lines = [x for x in text.strip().splitlines() if x.strip()]
    return lines[-1].strip() if lines else ""


def _final_thought(text):
    """MedGemma thinking models wrap reasoning in <unused94>...<unused95>; answer follows the close."""
    if "<unused95>" in text:
        return text.split("<unused95>", 1)[1], True
    if "<unused94>" in text:
        return text, False  # still inside the thought block when generation ended
    return text, True


def looks_truncated(text, max_tokens=500):
    t = text.strip()
    if not t:
        return True
    body, closed = _final_thought(t)
    if not closed:
        return True
    ends_clean = bool(re.search(r"[\.\!\?\)\*\$\"']\s*$|\b[A-E]\s*$", t))
    # ~4 chars/token; a response near the cap that does not end cleanly was cut off
    return (len(t) > 0.8 * max_tokens * 4 * 0.9) and not ends_clean


NEGATION = re.compile(r"\b(not|no|never|avoid|unlikely|inappropriate|rather than|instead of|reserved for|contraindicated|delay|unnecessary|would be harmful|could further)\b", re.I)


def _option_text_match(tail, options):
    """Return a letter if exactly one option's text appears in `tail` (closing sentences)."""
    if not options:
        return None
    if NEGATION.search(tail):  # a negated mention is not a commitment: leave for semantic adjudication
        return None
    tail_l = tail.lower()
    hits = []
    for k, v in options.items():
        v_l = str(v).lower().strip().rstrip(".")
        if len(v_l) >= 4 and v_l in tail_l:
            hits.append(k)
    return hits[0] if len(hits) == 1 else None


def parse_response(text, options=None, max_tokens=500):
    """-> (letter | None, status)"""
    if text is None or not str(text).strip():
        return None, "unparseable"
    text = str(text)
    body, closed = _final_thought(text)
    scope = body if closed else text

    # 1. explicit statements: take the LAST match so a conclusion beats earlier discussion
    for pat in EXPLICIT_PATTERNS:
        ms = list(pat.finditer(scope))
        if ms:
            return ms[-1].group(1).upper(), "explicit"

    # 2. bare letter / "D) text" on the final line
    m = FINAL_LINE.match(_last_line(scope))
    if m:
        return m.group(1).upper(), "final_line"

    # 3. closing sentences name exactly one option
    sents = [s for s in re.split(r"(?<=[\.\!\?])\s+|\n+", scope.strip()) if s.strip()]
    for k in (1, 2):  # final sentence, else final two; accept only an unambiguous single option
        letter = _option_text_match(" ".join(sents[-k:]), options)
        if letter:
            return letter, "option_text"

    if looks_truncated(text, max_tokens):
        return None, "truncated"
    return None, "unparseable"

In [6]:
rows = []
for r in raw:
    g = by_item[r["item"]]
    if r["finish"] == "error":
        letter, status = None, "api_error"
    else:
        letter, status = parse_response(r["response"], g["options"], MAX_TOKENS)
        if r["finish"] == "length" and letter is None: status = "truncated"
    if letter is None:              outcome = "unresolved"
    elif letter == g["answer_idx"]: outcome = "correct"
    elif letter in g["esc"]:        outcome = "wrong_but_escalates"
    else:                           outcome = "failure"
    rows.append(dict(model=r["model"], cond=r["cond"], item=r["item"], letter=letter, status=status, outcome=outcome,
                     finish=r["finish"], out_tokens=r["out_tokens"], response=r["response"]))
df_raw = pd.DataFrame(rows)                # untouched parser result: manual overrides are always applied to this
df_raw["group"] = np.where(df_raw.model.str.startswith("legacy_"), "legacy", "frontier")
df_raw["cell"] = df_raw.letter.fillna("-") + ":" + df_raw.outcome.map({"correct": "OK", "wrong_but_escalates": "esc", "failure": "FAIL", "unresolved": "??"})
df = df_raw.copy()
print(len(df), "rows scored |", df.model.nunique(), "models |", df.groupby("group").model.nunique().to_dict())

1404 rows scored | 13 models | {'frontier': 6, 'legacy': 7}


In [7]:
print("== parse status per condition ==")
print(df.groupby(["cond", "status"]).size().unstack(fill_value=0).to_string())

bad = df[df.status.isin(["truncated", "unparseable", "api_error"])]
print("\n== rows needing attention ==")
print(bad[["model", "cond", "item", "status", "finish", "out_tokens"]].to_string(index=False) if len(bad) else "none: every response parsed to a letter")

print("\n== outcome counts per model (n = items per cell) ==")
oc = df.pivot_table(index="model", columns=["cond", "outcome"], values="item", aggfunc="count", fill_value=0)
print(oc.to_string())

print("\n== pooled over all models ==")
pool = df.groupby(["cond", "outcome"]).size().unstack(fill_value=0)
pool["n"] = pool.sum(axis=1)
print(pool.to_string())

== parse status per condition ==
status  api_error  explicit  final_line  option_text  truncated  unparseable
cond                                                                        
cot            82       544           6            1         68            1
letter         58        40         562            0         42            0

== rows needing attention ==
                model   cond  item      status finish  out_tokens
     claude_haiku_5_5    cot    33   truncated length      1500.0
  deepseek_v4_1_flash    cot     8   truncated length      1500.0
  deepseek_v4_1_flash    cot    11   truncated length      1500.0
  deepseek_v4_1_flash    cot    12   truncated length      1500.0
  deepseek_v4_1_flash    cot    28   truncated length      1500.0
  deepseek_v4_1_flash    cot    29   truncated length      1500.0
  deepseek_v4_1_flash    cot    31   truncated length      1500.0
  deepseek_v4_1_flash    cot    32   truncated length      1500.0
  deepseek_v4_1_flash    cot    35 

In [8]:
w = df.pivot_table(index=["model", "item"], columns="cond", values="outcome", aggfunc="first").dropna().reset_index()
w["changed"] = w.letter != w.cot
print("== models whose outcome changed under CoT, per item ==")
print(w.groupby("item").changed.agg(changed="sum", of="count").to_string())
print("\n== transitions letter-only -> CoT ==")
print(w[w.changed].groupby(["letter", "cot"]).size().sort_values(ascending=False).to_string() or "none")

L = df.pivot_table(index=["model", "item"], columns="cond", values="letter", aggfunc="first").dropna(how="all")
print("\nsame letter chosen in both conditions:", int((L["letter"] == L["cot"]).sum()), "of", len(L))

== models whose outcome changed under CoT, per item ==
      changed  of
item             
0           3  13
1           2  13
2           1  13
3           1  13
4           3  13
5           1  13
6           3  13
7           2  13
8           3  13
9           2  13
10          2  13
11          4  13
12          3  13
13          0  13
14          0  13
15          3  13
16          2  13
17          1  13
18          1  13
19          2  13
20          2  13
21          1  13
22          1  13
23          5  13
24          1  13
25          1  13
26          2  13
27          1  13
28          4  13
29          1  13
30          1  13
31          6  13
32          1  13
33          2  13
34          0  13
35          2  13
36          1  13
37          0  13
38          1  13
39          1  13
40          2  13
41          3  13
42          3  13
43          2  13
44          2  13
45          0  13
46          1  13
47          4  13
48          2  13
49          4  13
50       